In [1]:
from pyspark.sql import functions as F
from functools import reduce
import re

WS = "ws-urban-analytics"
BRONZE = f"abfss://{WS}@onelake.dfs.fabric.microsoft.com/lh_bronze.Lakehouse"

StatementMeta(, 05e672a6-79fa-4d2f-ba97-ff335f70592c, 3, Finished, Available, Finished, False)

In [2]:
files = sorted(f.path for f in notebookutils.fs.ls(f"{BRONZE}/Files/raw/taxi") if f.name.endswith(".parquet"))
print(f"Files found: {len(files)}")
def load_month(path):
    year, month = map(int, re.search(r"(\d{4})-(\d{2})\.parquet$", path).groups())
    df = spark.read.parquet(path)
    cols = {c.lower(): c for c in df.columns}
    def c(name, dtype):
        return F.col(cols[name]).cast(dtype) if name in cols else F.lit(None).cast(dtype)
    return df.select(
        c("vendorid", "int").alias("vendor_id"),
        c("tpep_pickup_datetime", "timestamp").alias("pickup_ts"),
        c("tpep_dropoff_datetime", "timestamp").alias("dropoff_ts"),
        c("passenger_count", "int").alias("passenger_count"),
        c("trip_distance", "double").alias("trip_distance_mi"),
        c("ratecodeid", "int").alias("rate_code_id"),
        c("pulocationid", "int").alias("pu_location_id"),
        c("dolocationid", "int").alias("do_location_id"),
        c("payment_type", "int").alias("payment_type"),
        c("fare_amount", "double").alias("fare_amount"),
        c("tip_amount", "double").alias("tip_amount"),
        c("tolls_amount", "double").alias("tolls_amount"),
        c("total_amount", "double").alias("total_amount"),
        c("congestion_surcharge", "double").alias("congestion_surcharge"),
        c("airport_fee", "double").alias("airport_fee"),
    ).withColumn("year", F.lit(year)).withColumn("month", F.lit(month))
raw = reduce(lambda a, b: a.unionByName(b), [load_month(p) for p in files])
raw_count = raw.count()
print(f"Raw rows: {raw_count:,}")

StatementMeta(, 05e672a6-79fa-4d2f-ba97-ff335f70592c, 4, Finished, Available, Finished, False)

Files found: 3
Raw rows: 9,554,778


In [3]:
clean = (
    raw
    .withColumn("trip_duration_min",
                (F.unix_timestamp("dropoff_ts") - F.unix_timestamp("pickup_ts")) / 60)
    .filter((F.year("pickup_ts") == F.col("year")) & (F.month("pickup_ts") == F.col("month")))
    .filter(F.col("trip_duration_min").between(1, 360))
    .filter((F.col("trip_distance_mi") > 0) & (F.col("trip_distance_mi") < 200))
    .filter((F.col("fare_amount") >= 0) & (F.col("total_amount") >= 0))
    .filter(F.col("pu_location_id").between(1, 265) & F.col("do_location_id").between(1, 265))
    .dropDuplicates(["vendor_id", "pickup_ts", "dropoff_ts", "pu_location_id",
                     "do_location_id", "trip_distance_mi", "total_amount"])
    .withColumn("pickup_date", F.to_date("pickup_ts"))
    .withColumn("pickup_hour", F.hour("pickup_ts"))
    .withColumn("pickup_dow", F.dayofweek("pickup_ts"))
    .withColumn("is_weekend", F.dayofweek("pickup_ts").isin(1, 7))
    .withColumn("_ingested_at", F.current_timestamp())
)

StatementMeta(, 05e672a6-79fa-4d2f-ba97-ff335f70592c, 5, Finished, Available, Finished, False)

In [4]:
(clean.write.format("delta")
      .mode("overwrite")
      .option("overwriteSchema", "true")
      .partitionBy("year", "month")
      .saveAsTable("silver_taxi_trips"))
clean_count = spark.table("silver_taxi_trips").count()
print(f"Raw: {raw_count:,} | Clean: {clean_count:,} | Removed: {raw_count - clean_count:,} "
      f"({(raw_count - clean_count) / raw_count:.1%})")

StatementMeta(, 05e672a6-79fa-4d2f-ba97-ff335f70592c, 6, Finished, Available, Finished, False)

Raw: 9,554,778 | Clean: 9,179,406 | Removed: 375,372 (3.9%)
